# Präambel

In [1]:
from sympy import *
from IPython.display import display, Math, Latex, Markdown
#init_session()
init_printing(use_unicode=True)

In [2]:
def md(s):
    display(Markdown(s))
    
def ausnutzung(p, stab=None):
    color = '#000'
    emoji = ''
    if p > 1:
        color = '#f00'
        emoji = '🔴'
    elif p > 0.7:
        color = '#0f0'
        emoji = '🟢'
    elif p < 0.7:
        color = '#00f'
        emoji = '🟡'
    #md(f'#### <span style="color:{color};">{int(p*100)}%</span>')
    qs = stab.querschnitt if stab else ''
    md(f'#### {emoji} {int(N(p, 2)*100)}% ({qs})')
    return p


def note(s):
    md(f'<div style="width=100%; display: flex; justify-content: end"><span style="background-color:lightgreen; padding: 0.2rem 0.4rem; margin-right: 3rem; text-align: right; font-weight: bold;">{s}</span></div>')
ausnutzung(13.37)
ausnutzung(0.238)
N(0.238, 2) * 100

#### 🔴 1338% ()

#### 🟡 23% ()

In [65]:
from dataclasses import dataclass
from decimal import Decimal
from functools import partial

@dataclass
class Einwirkungen:
    N: float = None
    V: float = None
    My: float = None
    Mz: float = None
    
    def um_achse(self, attr, achse):
        return self.__getattribute__(f'{attr}{achse}')
        
    M = lambda self, achse: self.um_achse('M', achse)


@dataclass
class Querschnitt:
    querschnitt: str
    A: float
    Avy: float
    Avz: float
    b: float
    h: float
    tf: float
    Iy: float
    Iz: float
    iy: float
    iz: float
    KLy: str
    KLz: str

    def um_achse(self, attr, achse):
        return self.__getattribute__(f'{attr}{achse}')

    I = lambda self, achse: self.um_achse('I', achse)
    i = lambda self, achse: self.um_achse('i', achse)
    KL = lambda self, achse: self.um_achse('KL', achse)
    Av = lambda self, achse: self.um_achse('Av', achse)

    @property
    def KL_BDK(self):
        if (self.h / self.b) <= 2:
            return 'b'
        else:
            return 'c'

    
    
@dataclass
class Stahlguete:
    guete: str
    E: int
    fy: float
    fu: float
    
@dataclass
class Stab(Querschnitt, Stahlguete):
    l: float
    Lcry: float = None
    Lcrz: float = None

    def __init__(self, guete, QS, **kwargs):
        for k,v in kwargs.items():
            self.__setattr__(k, v)
            
        Querschnitt.__init__(self, **QS.__dict__)
        Stahlguete.__init__(self, **guete.__dict__)
        
    def um_achse(self, attr, achse):
        return self.__getattribute__(f'{attr}{achse}')
        
    Lcr = lambda self, achse: self.um_achse('Lcr', achse)

S235 = Stahlguete(guete='S235', E=21000, fy=23.5, fu=36)

HEB160 = Querschnitt(
    querschnitt='HEB 160',
    A   = 54.25,
    Avy = 41.6,
    Avz = 12.65,
    b   = 16,
    h   = 16,
    tf  = 1.3,
    Iy  = 2492,
    Iz  = 889.2,
    iy  = 6.78,
    iz  = 4.05,
    KLy = 'b',
    KLz = 'c',
)

HEB240 = Querschnitt(
    querschnitt='HEB 240',
    A   = 106,
    Avy = 81.6,
    Avz = 33.23,
    b   = 24,
    h   = 24,
    tf  = 1.7,
    Iy  = 11260,
    Iz  = 3923,
    iy  = 10.31,
    iz  = 6.08,
    KLy = 'b',
    KLz = 'c',
)
HEB300 = Querschnitt(
    querschnitt='HEB 300',
    A   = 149.1,
    Avy = 118.18,
    Avz = 47.45,
    b   = 30,
    h   = 30,
    tf  = 1.9,
    Iy  = 25170,
    Iz  = 8563,
    iy  = 12.99,
    iz  = 7.58,
    KLy = 'b',
    KLz = 'c',
)
HEB320 = Querschnitt(
    querschnitt='HEB 320',
    A   = 161.3,
    Avy = 127.43,
    Avz = 51.73,
    b   = 30,
    h   = 32,
    tf  = 2.05,
    Iy  = 30820,
    Iz  = 9239,
    iy  = 13.82,
    iz  = 7.57,
    KLy = 'b',
    KLz = 'c',
)
HEB500 = Querschnitt(
    querschnitt='HEB 500',
    A   = 238.6,
    Avy = 174.02,
    Avz = 89.78,
    b   = 30,
    h   = 50,
    tf  = 2.8,
    Iy  = 107200,
    Iz  = 12620,
    iy  = 21.19,
    iz  = 7.27,
    KLy = 'a',
    KLz = 'b',
)
HEB650 = Querschnitt(
    querschnitt='HEB 650',
    A   = 286,
    Avy = 186,
    Avz = 122.04,
    b   = 30,
    h   = 65,
    tf  = 3.1,
    Iy  = 210600,
    Iz  = 13980,
    iy  = 27.1,
    iz  = 6.99,
    KLy = 'a',
    KLz = 'b',
)

## Querschnitte

### HE-B

In [4]:
HEB160 = Querschnitt(
    querschnitt='HEB 160',
    A   = 54.25,
    Avy = 41.6,
    Avz = 12.65,
    b   = 16,
    h   = 16,
    tf  = 1.3,
    Iy  = 2492,
    Iz  = 889.2,
    iy  = 6.78,
    iz  = 4.05,
    KLy = 'b',
    KLz = 'c',
)

HEB240 = Querschnitt(
    querschnitt='HEB 240',
    A   = 106,
    Avy = 84.7,
    Avz = 33.24,
    b   = 24,
    h   = 24,
    tf  = 1.7,
    Iy  = 11260,
    Iz  = 3923,
    iy  = 10.31,
    iz  = 6.08,
    KLy = 'b',
    KLz = 'c',
)

HEB260 = Querschnitt(
    querschnitt='HEB 260',
    A   = 118.4,
    Avy = 94.4,
    Avz = 37.55,
    b   = 26,
    h   = 26,
    tf  = 1.75,
    Iy  = 14920,
    Iz  = 5135,
    iy  = 11.22,
    iz  = 6.58,
    KLy = 'b',
    KLz = 'c',
)
HEB650 = Querschnitt(
    querschnitt='HEB 650',
    A   = 286,
    Avy = 186,
    Avz = 122.04,
    b   = 30,
    h   = 65,
    tf  = 3.1,
    Iy  = 210600,
    Iz  = 13980,
    iy  = 27.1,
    iz  = 6.99,
    KLy = 'a',
    KLz = 'b',
)




### HE-A

In [5]:
HEA120 = Querschnitt(
    querschnitt='HEA 120',
    A   = 25.34,
    Avy = 20.05,
    Avz = 8.46,
    b   = 12,
    h   = 11.4,
    tf  = 0.8,
    Iy  = 606.2,
    Iz  = 230.9,
    iy  = 4.89,
    iz  = 3.02,
    KLy = 'b',
    KLz = 'c',
)
HEA140 = Querschnitt(
    querschnitt='HEA 140',
    A   = 31.42,
    Avy = 19.83,
    Avz = 6.25,
    b   = 14,
    h   = 13.3,
    tf  = 0.85,
    Iy  = 1033,
    Iz  = 389.3,
    iy  = 5.73,
    iz  = 3.52,
    KLy = 'b',
    KLz = 'c',
)
HEA160 = Querschnitt(
    querschnitt='HEA 160',
    A   = 38.77,
    Avy = 30.06,
    Avz = 13.21,
    b   = 16,
    h   = 15.2,
    tf  = 0.9,
    Iy  = 1673,
    Iz  = 615.6,
    iy  = 6.57,
    iz  = 3.98,
    KLy = 'b',
    KLz = 'c',
)
HEA200 = Querschnitt(
    querschnitt='HEA 200',
    A   = 53.83,
    Avy = 41.59,
    Avz = 18.08,
    b   = 20,
    h   = 19,
    tf  = 1,
    Iy  = 3692,
    Iz  = 1336,
    iy  = 8.28,
    iz  = 4.98,
    KLy = 'b',
    KLz = 'c',
)

### IPE

In [6]:
IPE200 = Querschnitt(
    querschnitt='IPE 200',
    A   = 28.50,
    Avy = 17,
    Avz = 14,
    b   = 10,
    h   = 20,
    tf  = 0.85,
    Iy  = 1940,
    Iz  = 142,
    iy  = 8.26,
    iz  = 2.24,
    KLy = 'a',
    KLz = 'b',
)

### RD

In [7]:
RD20 = Querschnitt(
    querschnitt='RD 20',
    A   = 3.14,
    Avy = 0,
    Avz = 0,
    b   = 2,
    h   = 2,
    tf  = 0,
    Iy  = 0,
    Iz  = 0,
    iy  = 0,
    iz  = 0,
    KLy = 'x',
    KLz = 'x',
)

RD30 = Querschnitt(
    querschnitt='RD 30',
    A   = 7.07,
    Avy = 0,
    Avz = 0,
    b   = 3,
    h   = 3,
    tf  = 0,
    Iy  = 0,
    Iz  = 0,
    iy  = 0,
    iz  = 0,
    KLy = 'x',
    KLz = 'x',
)

# Nachweis-Funktionen

## Querschnitte

### Zug

In [8]:
def zug(stab, last, A_net=None):
    γ_M0 = 1
    γ_M2 = 1.25
    N_plRd = (stab.A * stab.fy)/γ_M0
    N_uRd = (0.9 * A_net * stab.fu)/γ_M2 if A_net else oo

    return ausnutzung(N(last.N/min(N_plRd, N_uRd), 3), stab)

### Druck

In [9]:
def druck(stab, last):
    γ_M0 = 1
    N_cRd = (stab.A * stab.fy)/γ_M0
    
    return ausnutzung(N(last.N/N_cRd, 3), stab)

### Biegung

In [10]:
def biegung(um_achse, stab, last):
    γ_M0 = 1
    h = stab.h if um_achse == 'y' else stab.b
    W_elmin = stab.I(um_achse)/(h/2)
    M_elRd = (W_elmin * stab.fy)/γ_M0
    
    return ausnutzung(N(last.My/M_elRd, 3), stab)

biegung_y = partial(biegung, 'y')
biegung_z = partial(biegung, 'z')

### Querkraft

In [11]:
def querkraft(um_achse, stab, last):
    γ_M0 = 1
    V_plRd = stab.Av(um_achse) * (f_y/(sqrt(3) * γ_M0))
    
    return ausnutzung(N(last.V/V_plRd, 3), stab)

querkraft_y = partial(querkraft, 'y')
querkraft_z = partial(querkraft, 'z')

### Kombi: M + N

In [12]:
def kombi_M_N(um_achse, stab, last):
    γ_M0 = 1
    V_plRd = stab.Av(um_achse) * (stab.fy/(sqrt(3) * γ_M0))
    if (last.V / V_plRd) <= 0.5:
        md('Kombinierter Nachweis nicht Notwendig')
        md('Zug/Druck')
        zug(stab, last)
        md('Biegung')
        biegung(um_achse, stab, last)
    else:
        return ausnutzung(druck(stab, last) + biegung(um_achse, stab, last))

## Bauteil

In [13]:
γ_M1 = 1 #TODO tripple-check

def λ1(stab):
    return pi * sqrt(stab.E/stab.fy)

def λyz(um_achse, stab):
    return stab.Lcr(um_achse)/(stab.i(um_achse) * λ1(stab))

def KL2α(KL: str):
    return {'a0': 0.13, 'a': 0.21, 'b': 0.34, 'c': 0.49, 'd': 0.76}.get(KL)
    
    
def χyz(um_achse, stab):
    λ = λyz(um_achse, stab)
    α = KL2α(stab.KL(um_achse))
    Φ = 0.5* (1 + α * (λ-0.2) + λ**2)
    return min(1, 1/(Φ + sqrt(Φ**2 - λ**2)))

### Biegeknicken

In [14]:
def biege_knicken(um_achse, stab, last: Einwirkungen):
    assert(stab.Lcr(um_achse))
    χ = χyz(um_achse, stab)
    N_Rd = stab.A * (stab.fy/γ_M1)
    W_y = stab.Iy/(stab.h * 0.5) ## CHECK past?
    My_Rd =  W_y * (stab.fy / γ_M1) # W_y passt?

    n_y = last.N / (χ * N_Rd)
    k_y = min((1 + (λyz(um_achse, stab) - 0.2) * n_y),
              (1 + 0.8 * n_y),
             )
    α = 0.6 if um_achse == 'z' else 1

    #C_my = max(0.6 + 0.4 * ψ,
    #           0,4)
    C_my = 0.9 # XXX auf sicheren seite(?)
    res = (last.N/(χ * N_Rd)) + (α * k_y * (C_my * last.My)/My_Rd)
    return ausnutzung(N(res, 5), stab)

biege_knicken_y = partial(biege_knicken, 'y')
biege_knicken_z = partial(biege_knicken, 'z')
note('Passt W_Y berechnung?')
note('Passt C_my = 0.9 annahme? -> 0.9 gilt nur bei beweglichen systemem (hier nicht). C_my richtig ausrechnen...')

<div style="width=100%; display: flex; justify-content: end"><span style="background-color:lightgreen; padding: 0.2rem 0.4rem; margin-right: 3rem; text-align: right; font-weight: bold;">Passt W_Y berechnung?</span></div>

<div style="width=100%; display: flex; justify-content: end"><span style="background-color:lightgreen; padding: 0.2rem 0.4rem; margin-right: 3rem; text-align: right; font-weight: bold;">Passt C_my = 0.9 annahme? -> 0.9 gilt nur bei beweglichen systemem (hier nicht). C_my richtig ausrechnen...</span></div>

### Biegedrillknicken

In [15]:
def χLT(stab): ## Gem. EC3 (6.3.2.3)
    λz = λyz('z', stab)
    
    k_c = 1 # Auf der sicheren Seite
    k_p = 0.9 * 1 / (1 + (1/20) * (λz/(stab.h/stab.tf))**2)**0.25
    λLT = k_c * k_p * λz
        
    λLT0 = 0.4  # Höchstwert (EC3 Seite 70)
    β = 0.75    # Mindestwert
    αLT = KL2α(stab.KL_BDK)
    
    ΦLT = 0.5 * ( 1 + αLT * (λLT - λLT0) + β * λLT**2)
    return min(1 / (ΦLT + sqrt(ΦLT**2 - β * λLT**2)),
               1,
               1 / (λLT**2),
              )
note('λ_z-strich ist das selbe wie ohne strich? (siehe formel (30) im EC3 NA)')

<div style="width=100%; display: flex; justify-content: end"><span style="background-color:lightgreen; padding: 0.2rem 0.4rem; margin-right: 3rem; text-align: right; font-weight: bold;">λ_z-strich ist das selbe wie ohne strich? (siehe formel (30) im EC3 NA)</span></div>

In [16]:
def BDK(um_achse, stab, last: Einwirkungen):
    assert(stab.Lcr(um_achse))
    χ = χyz(um_achse, stab)
    N_Rd = stab.A * (stab.fy/γ_M1)
    W_y = stab.Iy/(stab.h * 0.5) ## CHECK past?
    My_Rd =  W_y * (stab.fy / γ_M1) # W_y passt?

    n_y = last.N / (χ * N_Rd)
    k_y = min((1 + (λyz(um_achse, stab) - 0.2) * n_y),
              (1 + 0.8 * n_y),
             )
    
    #C_my = max(0.6 + 0.4 * ψ,
    #           0,4)
    C_my = 0.9 # XXX auf sicheren seite(?)
    C_MLT = 0.9 # XXX auf sicheren seite?

    n_z = last.N / (χyz('z', stab) * N_Rd)
    k_LT = 1 - (0.1 * n_z)/(C_MLT - 0.25) ## ??? seite

    interaktionsbeiwert = {'y': k_y * C_my,
                           'z': k_LT,
                          }.get(um_achse)

    res = (last.N/(χ * N_Rd)) + (interaktionsbeiwert * last.My/(χLT(stab) * My_Rd))
    return ausnutzung(N(res, 5), stab)

BDK_y = partial(BDK, 'y')
BDK_z = partial(BDK, 'z')
note('C_MLT und C_my: sichere werte so ok?')

<div style="width=100%; display: flex; justify-content: end"><span style="background-color:lightgreen; padding: 0.2rem 0.4rem; margin-right: 3rem; text-align: right; font-weight: bold;">C_MLT und C_my: sichere werte so ok?</span></div>

# Stäbe

## 1 Pfette

In [17]:
pfette = Stab(S235, HEA200, l=6750, Lcry=6750, Lcrz=6750)

### Biegeknicken (um y)

In [18]:
# Schnee + wind(0):   N  = 16.87 kN 
# Schnee + wind(0):   My = 74.01 kN/m
biege_knicken_y(pfette, Einwirkungen(N=16.87, My=74.01))

#### 🔴 105% (HEA 200)

### Querschnitt (Druck & Schub)

In [19]:
# TODO

## 2 Binder Obergurt

In [20]:
obergurt = Stab(S235, HEB240, l=250.3, Lcry=250.3, Lcrz=250.3)

last_OG_zug = Einwirkungen(
    N  = 1718.15, # Schnee leitend + Wind(0)
    My = 71.32,   # Schnee leitend + Wind(0)
    V  = 25.51,   # Schnee leitend + Wind(0)
)
last_OG_druck = Einwirkungen(
    N  = 350.15,  # Wind(180)
    My = 12.79,   # Wind(180)
    V  = 5.70,    # Wind(180)
)

### Querschnitt (Zug)

In [21]:
kombi_M_N('y', obergurt, last_OG_zug)
kombi_M_N('z', obergurt, last_OG_zug)

Kombinierter Nachweis nicht Notwendig

Zug/Druck

#### 🟡 69% (HEB 240)

Biegung

#### 🟡 0% (HEB 240)

Kombinierter Nachweis nicht Notwendig

Zug/Druck

#### 🟡 69% (HEB 240)

Biegung

#### 🟡 0% (HEB 240)

### Querschnitt (Druck)

In [22]:
kombi_M_N('y', obergurt, last_OG_druck)
kombi_M_N('z', obergurt, last_OG_druck)

Kombinierter Nachweis nicht Notwendig

Zug/Druck

#### 🟡 14% (HEB 240)

Biegung

#### 🟡 0% (HEB 240)

Kombinierter Nachweis nicht Notwendig

Zug/Druck

#### 🟡 14% (HEB 240)

Biegung

#### 🟡 0% (HEB 240)

### Knicken in Ebene (y)

In [23]:
biege_knicken_y(obergurt, last_OG_druck)

#### 🟡 14% (HEB 240)

### Knicken aus der Ebene (z)

In [24]:
biege_knicken_z(obergurt, last_OG_druck)

#### 🟡 16% (HEB 240)

## 3 Binder Untergurt

In [25]:
#NB.: Laengsten stab mit hoechster N(x) kombiniert (sichere seite)
# Aus Ebene gibts keine Knicksicherung, gestamter stab wirkt

untergurt_feld =  Stab(S235, HEB260, l=378.1, Lcry=240, Lcrz=2*378.1)

last_UG_druck_feld = Einwirkungen(
    N  = 2092.02,  # Schnee leitend + Wind(0)
    My = 33.94,   # Schnee leitend + Wind(0)
    V  = 25.35,   # Schnee leitend + Wind(0)
)

last_UG_zug_feld = Einwirkungen(
    N  = 425.06,  # Wind(180)
    My = 6.67,   # Wind(180)
    V  = 5.4,   # Wind(180)
)


## Kragarm

# Variante 1
untergurt_krag_li = Stab(S235, HEB260, l=256, Lcry=256, Lcrz=2*648.7)
untergurt_krag_re = Stab(S235, HEB240, l=256, Lcry=256, Lcrz=2*485.8)

last_UG_druck_li = Einwirkungen(
    N  = 1909.6,  # Schnee leitend + Wind(0)
    My = 24.75,   # Schnee leitend + Wind(0)
    V  = 18.67,   # Schnee leitend + Wind(0)
)
last_UG_druck_re = Einwirkungen(
    N  = 463.77,  # Schnee leitend + Wind(0)
    My = 9.98,   # Schnee leitend + Wind(0)
    V  = 4.51,   # Schnee leitend + Wind(0)
)
last_UG_zug = Einwirkungen(
    N  = 325.39,  # Wind(180)
    My = 4.03,   # Wind(180)
    V  = 4.53,    # Wind(180)
)

# # Variante 2
# untergurt_krag_li = Stab(S235, HEB240, l=256, Lcry=256, Lcrz=2*138.3)
# untergurt_krag_re = Stab(S235, HEB240, l=256, Lcry=256, Lcrz=996.2)
# 
# last_UG_druck_li = Einwirkungen(
#     N  = 1909.6,  # Schnee leitend + Wind(0)
#     My = 24.75,   # Schnee leitend + Wind(0)
#     V  = 18.67,   # Schnee leitend + Wind(0)
# )
# last_UG_druck_re = Einwirkungen(
#     N  = 1391.5,  # Schnee leitend + Wind(0)
#     My = 24.75,   # Schnee leitend + Wind(0)
#     V  = 4.51,   # Schnee leitend + Wind(0)
# )
# last_UG_zug = Einwirkungen(
#     N  = 325.39,  # Wind(180)
#     My = 4.03,   # Wind(180)
#     V  = 4.53,    # Wind(180)
# )

### Querschnitt (Zug)

In [26]:
md('#### Kargarm (zug) in Ebene')
kombi_M_N('y', untergurt_krag_li, last_UG_zug)

md('#### Kargarm (zug) aus Ebene')
kombi_M_N('z', untergurt_krag_li, last_UG_zug)

md('#### Feld (zug) in Ebene')
kombi_M_N('y', untergurt_feld, last_UG_zug_feld)
md('#### Feld (zug) aus Ebene')
kombi_M_N('z', untergurt_feld, last_UG_zug_feld)

#### Kargarm (zug) in Ebene

Kombinierter Nachweis nicht Notwendig

Zug/Druck

#### 🟡 11% (HEB 260)

Biegung

...

Kombinierter Nachweis nicht Notwendig

Zug/Druck

#### 🟡 15% (HEB 260)

Biegung

#### 🟡 0% (HEB 260)

### Querschnitt (Druck)

In [27]:
md('#### Kragarm (druck) in Ebene')
kombi_M_N('y', untergurt_krag_li, last_UG_druck_li)
kombi_M_N('y', untergurt_krag_re, last_UG_druck_re)

md('#### Kragarm (druck) aus Ebene')
kombi_M_N('z', untergurt_krag_li, last_UG_druck_li)
kombi_M_N('z', untergurt_krag_re, last_UG_druck_re)

#### Kragarm (druck) in Ebene

Kombinierter Nachweis nicht Notwendig

Zug/Druck

#### 🟡 68% (HEB 260)

Biegung

...

Kombinierter Nachweis nicht Notwendig

Zug/Druck

#### 🟡 18% (HEB 240)

Biegung

#### 🟡 0% (HEB 240)

### Knicken in Ebene

In [28]:
md('#### Kragarm BDK')
BDK_y(untergurt_krag_li, last_UG_druck_li)
# BDK_y(untergurt_krag_2, last_UG_druck_2)  -> NICHT MASSGEBLICH

md('#### Feld BDK')
BDK_y(untergurt_feld, last_UG_druck_feld)

#### Kragarm BDK

#### 🟡 70% (HEB 260)

#### Feld BDK

#### 🟢 76% (HEB 260)

### BDK aus der Ebene

In [29]:
md('#### Kragarm BDK')
BDK_z(untergurt_krag_li, last_UG_druck_li)
BDK_z(untergurt_krag_re, last_UG_druck_re)

md('#### Feld BDK')
BDK_z(untergurt_feld, last_UG_druck_feld)


#### Kragarm BDK

#### 🔴 380% (HEB 260)

#### 🟢 72% (HEB 240)

#### Feld BDK

#### 🔴 178% (HEB 260)

In [30]:
# TODO

## 4 Binder: Streben

In [31]:
strebe_1_druck = Stab(S235, HEA140, l=192, Lcry=192, Lcrz=192)
strebe_1_zug   = Stab(S235, HEA140, l=192)

strebe_UG_druck = Stab(S235, HEB240, l=185, Lcry=185, Lcrz=185 * 2)
strebe_UG_zug   = Stab(S235, HEB240, l=185)


last_STR_1_druck = Einwirkungen(
    N  = 548,  # Schnee leitend + Wind(0)
    My = 0,
    V  = 0,
)
last_STR_1_zug = Einwirkungen(
    N  = 628,  # Wind(180)
    My = 0,
    V  = 0,
)

last_STR_UG_druck = Einwirkungen(
    N  = 1001.3,  # Schnee leitend + Wind(0)
    My = 0,
    V  = 0,
)
last_STR_UG_zug = Einwirkungen(
    N  = 207.05,  # Wind(180)
    My = 0,
    V  = 0,
)

### Querschnitt (Zug)

In [32]:
zug(strebe_1_zug, last_STR_1_zug)
note('hohe ausnutzung ok hier, weil keine schraubenloecher?')
zug(strebe_UG_zug, last_STR_UG_zug)

#### 🟢 85% (HEA 140)

<div style="width=100%; display: flex; justify-content: end"><span style="background-color:lightgreen; padding: 0.2rem 0.4rem; margin-right: 3rem; text-align: right; font-weight: bold;">hohe ausnutzung ok hier, weil keine schraubenloecher?</span></div>

#### 🟡 8% (HEB 240)

### Knicken in Ebene

In [33]:
## Druck: Knicken aus Ebene
# Maßgeblicher Stab:
# LF: Schnee leitend + Wind (0)
biege_knicken_y(strebe_1_druck, last_STR_1_druck)
biege_knicken_y(strebe_UG_druck, last_STR_UG_druck)

#### 🟢 78% (HEA 140)

#### 🟡 40% (HEB 240)

### Knicken aus der Ebene

In [34]:
## Druck: Knicken aus Ebene
# Maßgeblicher Stab:
# LF: Schnee leitend + Wind (0)
biege_knicken_z(strebe_1_druck, last_STR_1_druck)
biege_knicken_z(strebe_UG_druck, last_STR_UG_druck)

#### 🟢 93% (HEA 140)

#### 🟡 53% (HEB 240)

## 5 Dachverband

In [35]:
verband_dach   = Stab(S235, RD20, l=720)
last_verband_dach = Einwirkungen(
    N  = 61.83,   # Wind (90) + Schnee
    My = 0,
    V  = 0,
)

### Querschnitt (Zug)

In [36]:
zug(verband_dach, last_verband_dach)

#### 🟢 83% (RD 20)

## 6 Wandverband

### Querschnitt (Zug)

In [37]:
# TODO

## 7 Stütze A

In [38]:
stuetzeA = Stab(S235, HEA140, l=650, Lcry=650, Lcrz=650)

last_SA_zug = Einwirkungen(
    N  = 545,    # Schnee + wind(0)
    My = 0,
    V  = 0,
)
last_SA_druck = Einwirkungen(
    N  = 118.3,  # Wind (80)
    My = 0,
    V  = 0,
)


### Querschnitt (Zug)

In [39]:
zug(stuetzeA, last_SA_zug)

#### 🟢 73% (HEA 140)

### Querschnitt (Druck)

In [40]:
druck(stuetzeA, last_SA_druck)

#### 🟡 16% (HEA 140)

### Knicken in Ebene

In [41]:
biege_knicken_y(stuetzeA, last_SA_druck)

#### 🟡 33% (HEA 140)

### Knicken aus der Ebene

In [42]:
biege_knicken_z(stuetzeA, last_SA_druck)

#### 🟢 79% (HEA 140)

## 8 Stütze B

In [66]:
stuetzeB = Stab(S235, HEB320, l=775, Lcrz=625)


last_SB_druck = Einwirkungen(
    N  = 1221,   # Schnee + wind(0)
    My = 220.38, # Schnee + wind(0)
    V  = 146.92,
)
last_SB_zug = Einwirkungen(
    N  = 221,    # Wind(180)
    My = 220.38, # Schnee + Wind(0)
    V  = 146.92, # Schnee + wind(0)
)


### Knicklänge (um y)

In [67]:
h_Fachwerk = 108                 # Achsabstand der Gurte in Mitte des Feldbereichs vom Fachwerk
z = h_Fachwerk * 0.5
l_R = 500
l = 775
l1 = stuetzeA.l

N_A = abs(last_SA_zug.N)
N_B = abs(last_SB_druck.N)
F = N_A * (l / l1)   # <--- Einsparungspotential!  ggf hoehe bis traegermitte waehlbar


EI = stuetzeB.E * stuetzeB.Iy
EI_R = obergurt.E * ((obergurt.Iy + obergurt.A * z**2) + (untergurt_feld.Iy + untergurt_feld.A * z**2))

display(stuetzeA.l)
display(EI_R)

ν = 1 / ( (l_R * EI)/(l * EI_R) + 1)
η = ( (3 * ν) / ( (0.216 * ν**2 + 1) * N_B + F)) * (EI/l**2)
stuetzeB.Lcry = N(pi * sqrt(EI/(η*N_B)))
stuetzeB

Stab(guete='S235', E=21000, fy=23.5, fu=36, querschnitt='HEB 320', A=161.3, Avy=127.43, Avz=51.73, b=30, h=32, tf=2.05, Iy=30820, Iz=9239, iy=13.82, iz=7.57, KLy='b', KLz='c', l=775, Lcry=1879.02449702529, Lcrz=625)

### Querschnitt (Zug)

In [68]:
kombi_M_N('y', stuetzeB, last_SB_zug)
kombi_M_N('z', stuetzeB, last_SB_zug)

Kombinierter Nachweis nicht Notwendig

Zug/Druck

#### 🟡 5% (HEB 320)

Biegung

#### 🟡 0% (HEB 320)

Kombinierter Nachweis nicht Notwendig

Zug/Druck

#### 🟡 5% (HEB 320)

Biegung

#### 🟡 1% (HEB 320)

### Querschnitt (Druck)

In [69]:
kombi_M_N('y', stuetzeB, last_SB_druck)
kombi_M_N('z', stuetzeB, last_SB_druck)

Kombinierter Nachweis nicht Notwendig

Zug/Druck

#### 🟡 32% (HEB 320)

Biegung

#### 🟡 0% (HEB 320)

Kombinierter Nachweis nicht Notwendig

Zug/Druck

#### 🟡 32% (HEB 320)

Biegung

#### 🟡 1% (HEB 320)

### BDK in Ebene

In [70]:
BDK_y(stuetzeB, last_SB_druck)

#### 🟢 89% (HEB 320)

### Knicken aus der Ebene

In [71]:
biege_knicken_z(stuetzeB, last_SB_druck)

#### 🟡 52% (HEB 320)

### Biegung

In [72]:
# TODO